In [ ]:
import os

ANTHROPIC_API_KEY = "ANTHROPIC_API_KEY"

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

TAVILY_API_KEY = "TAVILY_API_KEY"

os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

In [ ]:
import asyncio, websockets, aiohttp, json, random
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_anthropic import ChatAnthropic
from langgraph_agent import LangGraphAgentState
from langgraph.graph import StateGraph
from langgraph.types import Command, interrupt

from config import AgentConfig, load_config
from chaos_agent import ChaosAgent
from dotenv import load_dotenv

import os
import asyncio
from langchain_core.tools import tool

In [ ]:
from typing import Dict, TypedDict, Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class LangGraphAgentState(TypedDict):
    name: str
    personality: str
    style: str
    stake_amount: int
    current_message: str
    messages: list
    decisions: list

In [ ]:
@tool
def human_assistance(query: str) -> str:
    """Request assistance from a human."""
    human_response = interrupt({"query": query})
    return human_response["data"]

def make_decision(state: LangGraphAgentState) -> LangGraphAgentState:
    ai_message = llm_with_tools.invoke(state.messages)
    
    state.decisions.append(ai_message.content)

    return state

tool = TavilySearchResults(max_results=2)
tools = [tool, human_assistance]
llm = ChatAnthropic(model="claude-3-5-sonnet-20240620")
llm_with_tools = llm.bind_tools(tools)

def create_graph() -> StateGraph:
    workflow = StateGraph(LangGraphAgentState)

    workflow.add_node("make_decision", make_decision)

    workflow.set_entry_point("make_decision")
    workflow.set_finish_point("make_decision")

    return workflow.compile()